# Import Libraries

In [1]:
from datasets import load_dataset, Features, Value, ClassLabel
from transformers import AutoProcessor, AutoModelForAudioClassification, Trainer, TrainingArguments, TrainerCallback
import librosa
import numpy as np
import torch
import json
from IPython.display import display
import os
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import DataLoader

# Prepare Dataset

In [2]:
class_names= ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
features = Features({'file': Value('string'), 'genre': ClassLabel(names=class_names)})

In [3]:
ds = load_dataset('marsyas/gtzan', name='en-US', features=features, split='train',trust_remote_code=True).train_test_split(test_size=0.2, stratify_by_column='genre')
ds

DatasetDict({
    train: Dataset({
        features: ['file', 'genre'],
        num_rows: 799
    })
    test: Dataset({
        features: ['file', 'genre'],
        num_rows: 200
    })
})

In [4]:
# Check distribution in both splits
print("Train distribution:", Counter(ds['train']['genre']))
print("Test distribution:", Counter(ds['test']['genre']))


Train distribution: Counter({8: 80, 2: 80, 7: 80, 1: 80, 4: 80, 9: 80, 0: 80, 6: 80, 3: 80, 5: 79})
Test distribution: Counter({2: 20, 0: 20, 3: 20, 9: 20, 8: 20, 5: 20, 7: 20, 6: 20, 1: 20, 4: 20})


In [6]:
features = ds['train'].features
features

{'file': Value(dtype='string', id=None),
 'genre': ClassLabel(names=['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'], id=None)}

### Create label to index file

In [7]:
id2label = {idx:features['genre'].int2str(idx) for idx in range(10)}
label2id = {value: key for key, value in id2label.items()}

label_idx = {'id2lable': id2label,
             'label2id': label2id
             }

with open('label_idx.json', 'w')as file:
    file.write(json.dumps(label_idx, indent=2))

In [8]:
with open('label_idx.json', 'r')as file:
    json_object = json.load(file)
    
label2id = json_object['label2id']
id2label = json_object['id2lable']
display(label2id, id2label)

{'blues': 0,
 'classical': 1,
 'country': 2,
 'disco': 3,
 'hiphop': 4,
 'jazz': 5,
 'metal': 6,
 'pop': 7,
 'reggae': 8,
 'rock': 9}

{'0': 'blues',
 '1': 'classical',
 '2': 'country',
 '3': 'disco',
 '4': 'hiphop',
 '5': 'jazz',
 '6': 'metal',
 '7': 'pop',
 '8': 'reggae',
 '9': 'rock'}

# Prepare Train & Test Data


### Define model name and device

In [2]:
model_name = "facebook/wav2vec2-large-960h-lv60-self"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Define Processor for Preprocessing

In [10]:
processor = AutoProcessor.from_pretrained(model_name)
def preprocess_format_audio(data, sampling_rate, max_dura):
    input_values = [] 
    for file_path in data['file']:
        try:
            audio, _ = librosa.load(file_path, sr=sampling_rate)
            audio =  audio[:max_dura]
            inputs = processor(audio,sampling_rate= sampling_rate, return_tensors="pt").input_values.squeeze(0)
            input_values.append(inputs.numpy())
        except Exception as e:
            print(f'Failed to process {file_path} : {e}')
    return {
        'input_values': input_values,
        'labels': data['genre']
    }
    

In [3]:
# Save embeddings
def save_embeddings(X, y, file_path):
    np.savez(file_path, input_values=X, labels=y)
    print(f"Saved {len(X)} processed audio embeddings to {file_path}.")
    
# Load saved embeddings
def load_embeddings(file_path):
    data = np.load(file_path, allow_pickle=True),
    print(f"Loaded {len(data['input_values'])} audio embeddings and {len(data['labels'])} labels.")
    return data["input_values"], data["labels"]

### Preprocess Raw Audio File 

In [12]:
sr = 16000  #use this for the wav2vec2 model sampling rate
max_duration = sr * 20 #max_duration = sampling rate * duration(in seconds)
train_dataset = preprocess_format_audio(ds['train'], sampling_rate=sr, max_dura= max_duration)
test_dataset = preprocess_format_audio(ds['test'],sampling_rate=sr, max_dura= max_duration)

In [ ]:
# Directory to save embeddings
train_embedding_save_path = "Embeddings-20secs"

if not os.path.exists(train_embedding_save_path):
    os.makedirs(train_embedding_save_path)
save_embeddings(X=train_dataset['input_values'], y= np.array(train_dataset['labels']), file_path='Embeddings-20secs\\train_dataset.npz')
save_embeddings(X=test_dataset['input_values'], y= np.array(test_dataset['labels']), file_path='Embeddings-20secs\\test_dataset.npz')


### Load the Processed Data

In [4]:
X_train, y_train = load_embeddings('Embeddings-20secs\\train_dataset.npz')
X_test, y_test = load_embeddings('Embeddings-20secs\\test_dataset.npz')

Loaded 799 audio embeddings and 799 labels.
Loaded 200 audio embeddings and 200 labels.


In [7]:
class EmbeddingDataset(torch.utils.data.Dataset):
    '''
    
        This class take processed input values and labels to create Dataset
        to match the Trainer API format.
    
    '''
    def __init__(self, input_values, labels):
        self.input_values = torch.tensor(input_values, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.input_values)

    def __getitem__(self, idx):
        return {"input_values": self.input_values[idx], "labels": self.labels[idx]}
    
class SaveAtSpecificEpochCallback(TrainerCallback):
    '''
    
        This callback class saves the model at specific checkpoint. 
        Change the output_dir to your desire directory.
    
    '''
    def __init__(self, save_epochs):
        self.save_epochs = save_epochs

    def on_epoch_end(self, args ,state, control, model=None, **kwargs):
        # Save the model at specific epochs
        if state.epoch in self.save_epochs:
            output_dir = f"Models-20secs-2/20secs-checkpoint-epoch-{int(state.epoch)}"
            model.save_pretrained(output_dir)
            print(f"Model saved at {output_dir} for epoch {int(state.epoch)}")


### Create Train and Test Dataset

In [8]:
train_dataset = EmbeddingDataset(X_train, y_train)
test_dataset = EmbeddingDataset(X_test, y_test)

In [9]:
print("Training set class distribution (after preprocessing):", Counter(train_dataset.labels.numpy()))
print("Test set class distribution (after preprocessing):", Counter(test_dataset.labels.numpy()))
print("Train Size: ",train_dataset.labels.shape)
print("Test Size: ",test_dataset.labels.shape)

Training set class distribution (after preprocessing): Counter({4: 80, 6: 80, 1: 80, 7: 80, 8: 80, 9: 80, 3: 80, 2: 80, 0: 80, 5: 79})
Test set class distribution (after preprocessing): Counter({9: 20, 3: 20, 1: 20, 8: 20, 7: 20, 0: 20, 6: 20, 4: 20, 2: 20, 5: 20})
Train Size:  torch.Size([799])
Test Size:  torch.Size([200])


# Train the Model
### Instantiate the Model

In [10]:
# Initialize the pre-trained Wav2Vec2 model
model = AutoModelForAudioClassification.from_pretrained(
    model_name,
    num_labels=10
)

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits to predicted class labels
    predictions = logits.argmax(axis=-1)

    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

### Set Hyperparameter

In [ ]:
#define step
batch_size = 2
# Training arguments
training_args = TrainingArguments(
    output_dir="./Checkpoints_20secs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=4, #use gradient accumalation to get the same effect as batch size = 8
    num_train_epochs=100,
    logging_dir="./logs",
    logging_steps= 100, #log training loss (if batch size and gradient accumalation is change, you need to change this too to get training loss at every epoch)
    save_total_limit=5, #save only 5 best model
    load_best_model_at_end=True,
    fp16=True,
    metric_for_best_model="accuracy",
    gradient_checkpointing=True,
    dataloader_pin_memory=True, # Ensures faster transfer to GPU
    weight_decay=0.01,  # Add L2 regularization
    max_grad_norm=1.0  #Add Gradient Clipping
)


# Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics, 
    tokenizer= None, #since we already processed the data, there's no need to use tokenizer here
    callbacks=[SaveAtSpecificEpochCallback(save_epochs=[10, 25, 50, 75, 100])]
)


In [ ]:
# Train the model
trainer.train()

# Evaluate the Model

In [ ]:
# Evaluate the model
trainer.evaluate()

### Plot Confusion Matrix

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=8)
model.eval()

predictions, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        # Directly use batch["input_values"] if it's already a Tensor
        inputs = batch["input_values"].to(device)
        labels = batch["labels"]

        outputs = model(inputs)
        preds = torch.argmax(outputs.logits, axis=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.numpy())

# Metrics
accuracy = accuracy_score(true_labels, predictions)
print(f"Accuracy: {accuracy}")
print(classification_report(true_labels, predictions, target_names=label2id.keys()))

# Confusion Matrix
cm = confusion_matrix(true_labels, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label2id.keys())

# Plot Confusion Matrix
plt.figure(figsize=(10, 7))
disp.plot(cmap=plt.cm.Blues, ax=plt.gca())
plt.title("Confusion Matrix")
plt.show()

### Plot Learning Curve

In [12]:
#Get log history
with open('Inference Models\model-20secs-checkpoint-5400\\trainer_state.json', 'r')as file: #you may change this into your model state
    json_object = json.load(file)

log_history = json_object['log_history']

In [ ]:
train_loss = []
val_loss = []

for data in log_history:
    for key, value in data.items():
        if key == 'loss':
            train_loss.append(value)
            break
        if key == 'eval_loss':
            val_loss.append(value)
            break

# Ensure the losses have the same length (in case there are skipped logs)
min_epochs = min(len(train_loss), len(val_loss))
train_loss = train_loss[:min_epochs]
val_loss = val_loss[:min_epochs]

# Plot the losses
epochs = range(1, min_epochs + 1)
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss, label="Training Loss", marker='o')
plt.plot(epochs, val_loss, label="Validation Loss", marker='x')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Learning Curve")
plt.legend()
plt.grid(True)
plt.show()


### Inference

In [21]:
model_name = "Inference Models\model-20secs-checkpoint-5400"
model = AutoModelForAudioClassification.from_pretrained(model_name)

In [ ]:
# Switch the model to evaluation mode
model.eval()

# Load the test dataset and DataLoader
test_loader = DataLoader(test_dataset, batch_size=1)
idx = 10  #change this to the sample index you want to use
# Evaluate a single example
with torch.no_grad():
    inputs = test_dataset.input_values[idx]
    inputs = inputs.clone().detach().unsqueeze(0).to(device)  # Add batch dimension
    # Pass through the model
    logits = model(inputs).logits

# Get predicted class
predicted_id = torch.argmax(logits, dim=-1).item()
print(f"Predicted class ID: {predicted_id}")
print(f"Predicted label: {id2label[str(predicted_id)]}")
print(f"True clss ID: {test_dataset.labels[idx]}")
